# Training data for a CAM physics kernel

`mmacro_pcond` -- the CAM5 cloud macrophysics -- runs here as an ordinary
Python function in a standalone image linked from the pinned iCESM object
code, so every number it answers is the model's own arithmetic.

Two cells.  The first takes real atmospheric columns out of a capture of
the running model; the second draws samples around them and lets the
Fortran answer each one.

Every path is resolved from `site.env` at the repository root (see the
README's *Site configuration*); nothing below names a user.

## 1. Real atmospheric columns

A capture holds every argument of every call `mmacro_pcond` received over a
50-step run on 512 ranks -- 691,300 live columns of the model's own state.
This draws `ANCHOR_COLUMNS` of them, all 35 arguments of a column taken
together so a sample stays one coherent atmospheric state.

Captures under your own scratch are found automatically.  To use one
somebody else published, set `FREECAM_CAPTURE` in `site.env` to the
directory holding it -- they are world-readable and large, and nothing has
to be built to read one.  Making a *new* capture is a larger undertaking --
a configured CESM case, its oracle run, a CAM executable rebuilt with the
capture patch, and a 512-rank job -- and is only worth it for a
configuration this one does not cover: a different namelist, resolution, or
routine.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
import numpy as np

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / 'pyproject.toml').is_file():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
from freecam import site

SCRATCH = site.resolved(repo=REPO)['scratch']
WORK = SCRATCH / 'pyCAM' / 'kernel-training'; WORK.mkdir(parents=True, exist_ok=True)

FUNCTION = 'mmacro_pcond'
ANCHOR_COLUMNS = int(os.environ.get('PYCAM_ANCHOR_COLUMNS', 200_000))

IMAGE = REPO / 'build/pi_cam_standalone' / FUNCTION / 'manifest.json'
assert IMAGE.is_file(), (
    f'no standalone image at {IMAGE}\n'
    f'  build one: tools/build_pi_cam_standalone_function.py --function {FUNCTION}')

# Everything comes out of a capture: the model's own arguments for every
# call it made over 50 steps on 512 ranks.  Anchors are an extract of it at
# whatever size is asked for, so ANCHOR_COLUMNS is the only knob.
PUBLISHED = site.path('FREECAM_CAPTURE', repo=REPO)

def live_columns(path):
    """How many real columns a capture holds -- the only thing that ranks them.

    A two-timestep capture and a fifty-step one are the same file to a glob
    and hold 28,000 states against 690,000.  Reading one small member decides
    which is which.
    """
    try:
        return int(np.asarray(np.load(path, allow_pickle=True)['ncol']).sum())
    except Exception:
        return -1

found = sorted(SCRATCH.glob(f'pyCAM/PI-cam/*/bundles/{FUNCTION}_capture.npz'))
if PUBLISHED is not None:
    found.append(PUBLISHED / f'{FUNCTION}_capture.npz')
candidates = sorted(((live_columns(path), path) for path in found if path.is_file()),
                    reverse=True)
assert candidates, (
    'no capture bundle.  Looked in:\n  ' + '\n  '.join(str(path) for path in found)
    + '\n  point FREECAM_CAPTURE in site.env at a published one,'
      '\n  or produce one: validation/jobs/submit.sh'
      ' validation/jobs/pi_cam_function_capture_training.pbs')
held, CAPTURE = candidates[0]
assert ANCHOR_COLUMNS <= held, (
    f'asked for {ANCHOR_COLUMNS:,} anchors; the best capture holds {held:,}')

ANCHORS = WORK / f'anchors_{ANCHOR_COLUMNS}.npz'
if not ANCHORS.is_file():
    print(f'extracting {ANCHOR_COLUMNS:,} anchors from {held:,} captured columns', flush=True)
    subprocess.run([sys.executable, str(REPO / 'tools/extract_pi_cam_anchor_columns.py'),
                    '--function', FUNCTION, '--bundle', str(CAPTURE),
                    '--columns', str(ANCHOR_COLUMNS), '--output', str(ANCHORS)],
                   check=True, stdout=subprocess.DEVNULL)

anchors = np.load(ANCHORS, allow_pickle=True)
where = json.loads(str(anchors['provenance']))
USING = where['columns']
print(f"{USING:,} anchors from {where['live_columns']:,} captured columns")
print(f'  capture: {CAPTURE}')
print(f"  {len([n for n in anchors.files if not n.startswith('meta_') and n != 'provenance'])} arguments each, "
      f"{np.asarray(anchors['t0']).shape[1]} levels")
print(f"  {ANCHORS}  ({ANCHORS.stat().st_size / 1e9:.2f} GB)")

## 2. Samples

Each sample draws one anchor column, perturbs it, draws the nine tunable
namelist parameters, and calls the Fortran for the answer.  The anchors are
drawn with replacement, so more samples than anchors means the same state
under several namelists -- which is what teaches the parameter response.

The work is split across processes because half a million samples in one
process is more memory than a login node gives.  At full size this belongs
in `validation/jobs/pi_cam_mmacro_dataset_build.pbs`, which runs this same
path and then trains on it.

In [ ]:
SAMPLES = int(os.environ.get('PYCAM_SAMPLES', 500_000))
CHUNKS = int(os.environ.get('PYCAM_CHUNKS', 10))
per_chunk = SAMPLES // CHUNKS
seconds = SAMPLES / 450 / CHUNKS         # about 450 columns a second per process
print(f'{SAMPLES:,} samples in {CHUNKS} processes of {per_chunk:,}  '
      f'(roughly {seconds / 60:.0f} min)' if seconds > 90 else
      f'{SAMPLES:,} samples in {CHUNKS} processes of {per_chunk:,}  '
      f'(roughly {seconds:.0f} s)', flush=True)

started = time.monotonic()
running, files = [], []
for index in range(CHUNKS):
    out = WORK / f'chunk_{index}.nc'
    files.append(out)
    if out.is_file():
        continue
    running.append(subprocess.Popen(
        [sys.executable, str(REPO / 'examples/generate_mmacro_pcond_dataset.py'),
         '--samples', str(per_chunk), '--seed', str(2026 + index),
         '--anchor-bundle', str(ANCHORS), '--output', str(out),
         # each process holds only its share of the anchors, so the memory
         # is the same whether one runs or ten
         '--anchor-part', str(index), '--anchor-parts', str(CHUNKS)],
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE))
for process in running:
    if process.wait() != 0:
        raise SystemExit(process.stderr.read().decode()[-2000:])
print(f'  generated in {time.monotonic() - started:.0f} s', flush=True)

TRAINING = WORK / 'training'
subprocess.run([sys.executable, str(REPO / 'tools/build_pi_cam_kernel_training_set.py'),
                '--function', FUNCTION, '--dataset', *[str(f) for f in files],
                '--output', str(TRAINING)], check=True, stdout=subprocess.DEVNULL)

X, Y = np.load(TRAINING / 'X.npy', mmap_mode='r'), np.load(TRAINING / 'Y.npy', mmap_mode='r')
meta = np.load(TRAINING / 'meta.npz', allow_pickle=True)
names = [str(n) for n in meta['x_names']]
print()
print(f'X {X.shape[0]:,} x {X.shape[1]}   Y {Y.shape[0]:,} x {Y.shape[1]}   -> {TRAINING}')
print(f"  {X.shape[1] - sum(n.startswith('parameter') for n in names)} state features"
      f" + {sum(n.startswith('parameter') for n in names)} namelist features")
reuse = X.shape[0] / USING
print(f'  drawn from {USING:,} anchors, so each atmospheric state appears '
      f'{reuse:.1f} times on average')
if reuse < 1.5:
    # Reuse is what teaches the namelist response: the same state under
    # several parameter draws is the only way to see the parameters move
    # while the state holds still.
    print(f'  -- below about 1.5 a state rarely gets a second namelist, so the'
          f' parameter\n     response is learned only across states.  Raise SAMPLES'
          f' or lower ANCHOR_COLUMNS.')

## What to do with it

```bash
# train, and score on held-out rows
tools/train_pi_cam_gated_surrogate.py --training $SCRATCH/pyCAM/kernel-training/training \\
    --output surrogate.pt --epochs 60
tools/evaluate_pi_cam_kernel_surrogate.py --model surrogate.pt \\
    --training $SCRATCH/pyCAM/kernel-training/training --output skill.json

# or the whole path -- anchors, samples, X and Y, train, score -- in one job
qsub -A <allocation> validation/jobs/pi_cam_mmacro_dataset_build.pbs
```

One caution.  Samples reuse anchors, so a *random* train/holdout split puts
variants of the same atmospheric state on both sides: at 500,000 samples
over 200,000 anchors, 91% of held-out samples have an anchor the model
trained on.  Split by anchor if the held-out score has to mean anything.